# WLS-Var investigation — single-peptide & collapse-vs-pooled

Builds on the point-level Monte-Carlo in `tests/benchmark/bench_linear_weights.py` with a
**peptide-level generative model** (protein × peptide × timepoint), which is what distinguishes
the two rollup methods. See `reports/2026-07-24_linear_wls_per_point_var.md` for the estimator
definitions (`ols` / `wls` / `irls` / `wls-var`) and the eBayes moderation.

**Model** (`wls_var_sim.simulate_protein`): each protein has `n_pep` peptides; each peptide gets an
intrinsic k-offset `~ N(0, tau)` **shared across conditions** (a peptide is measured in both control
and treatment, so the offset cancels in the paired Δk). θ(t) = 1−e^{−k·e^{offset}·t} + N(0, σ_pep);
σ_pep is heteroscedastic; the *estimate* σ̂ is noisy at `df = var_df`.

**Two rollups.** *collapse* = one inverse-variance θ per (biorep, t) cell → `Var(θ_i)=1/Σ(1/σ²)`
(under-states via the between-peptide dispersion). *pooled* = every peptide-point is its own point →
`Var = σ̂²_pep` (unbiased, no dispersion) but pseudoreplicated.

Metric: `reject` = the Δk-test rejection rate (**Type-I** when kA=kB, **power** otherwise). k_A
absolute coverage is *not* reported — it is contaminated by the peptide-sampling bias that cancels
in Δk, so it is not the inferential target.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from wls_var_sim import run, d0_grid, fit_fdist, simulate_protein, collapse
pd.set_option("display.width", 140)
NSIM = 300   # bump for tighter estimates

## Scenario 2 — collapse vs pooled (multi-peptide, `n_pep=4`)

Does the rollup method change how `wls-var` behaves? (Your insight: pooled keeps the *unbiased*
per-peptide variance and sidesteps the collapse's dispersion, at the cost of pseudoreplication.)

In [2]:
print("TYPE-I  (kA = kB = 0.10)")
display(run(0.10, 0.10, n_pep=4, nsim=NSIM, seed=1))
print("\nPOWER   (kA = 0.10, kB = 0.14)")
display(run(0.10, 0.14, n_pep=4, nsim=NSIM, seed=3))

TYPE-I  (kA = kB = 0.10)


,method,rollup,n,dk_bias,dk_rmse,reject
0,wls_fit,collapse,300,0.000227,0.002992,0.043333
1,wls_fit,pooled,300,-0.000285,0.003785,0.003333
2,wls_fit_var,collapse,300,0.000349,0.004007,0.110000
3,wls_fit_var,pooled,300,0.000362,0.004660,0.093333
4,ebayes,collapse,300,0.000324,0.003539,0.080000
5,ebayes,pooled,300,0.000118,0.003110,0.003333



POWER   (kA = 0.10, kB = 0.14)


,method,rollup,n,dk_bias,dk_rmse,reject
0,wls_fit,collapse,300,0.000424,0.008352,1.000000
1,wls_fit,pooled,300,0.002942,0.008436,0.996667
2,wls_fit_var,collapse,300,0.000542,0.008972,1.000000
3,wls_fit_var,pooled,300,0.001956,0.009521,1.000000
4,ebayes,collapse,300,0.000481,0.008711,1.000000
5,ebayes,pooled,300,0.002150,0.007871,1.000000


**Reading.** `wls_fit` (shipped) is ~nominal under collapse and conservative under pooled; raw
`wls_fit_var` inflates Type-I under **both** (noisy weights); **eBayes fixes it, and pooled+eBayes is
the safest** cell. Power stays ≈1.0 at Δk=0.04 for all — the conservatism costs no power on a real
effect. → **the rollup method interacts strongly with `wls-var`; pooled may be the better substrate.**

## Scenario 1 — single-peptide proteins (`n_pep=1`)

The few-peptide case: low df, no *between*-peptide heterogeneity (collapse ≡ pooled), so the only
variation `wls-var` can exploit is per-timepoint. Is there any gain?

In [3]:
display(run(0.10, 0.10, n_pep=1, nsim=NSIM, seed=2))   # TYPE-I

,method,rollup,n,dk_bias,dk_rmse,reject
0,wls_fit,collapse,300,0.000124,0.006261,0.026667
1,wls_fit,pooled,300,0.000124,0.006261,0.026667
2,wls_fit_var,collapse,300,0.000391,0.007736,0.110000
3,wls_fit_var,pooled,300,0.000391,0.007736,0.110000
4,ebayes,collapse,300,0.000296,0.006818,0.043333
5,ebayes,pooled,300,0.000296,0.006818,0.043333


**Reading.** Per-point weighting **hurts** here — `dk_rmse` rises `wls_fit` → `wls_fit_var`
(≈0.006 → 0.008) and Type-I inflates (≈0.04 → 0.12) with *no* efficiency to gain. **eBayes shrinks it
back toward `wls_fit`** (RMSE ≈0.007, Type-I ≈0.06). So for single-/few-peptide proteins the
moderated estimator correctly does ≈nothing — protecting the challenging cases, as hoped.

## d₀ frontier + Smyth `fitFDist`

Is the data-estimated d₀ (limma `fitFDist`) optimal, or just sensible? Sweep d₀ for the eBayes arm
(collapse) and compare to what `fitFDist` returns on the same variance estimates.

In [4]:
print("eBayes d0 sweep (collapse, TYPE-I at kA=kB=0.10)")
display(d0_grid(0.10, 0.10, n_pep=4, nsim=NSIM, seed=1, d0s=(0.5, 1, 2, 4, 8)))

rng = np.random.default_rng(9); s2, dfs = [], []
for _ in range(400):
    c = collapse(simulate_protein(0.10, 0.10, 4, rng)); s2 += c[:, 3].tolist(); dfs += c[:, 4].tolist()
d0_hat, s0 = fit_fdist(np.array(s2), np.array(dfs))
print(f"fitFDist → d0 = {d0_hat:.2f},  s0^2 = {s0:.2e}")

eBayes d0 sweep (collapse, TYPE-I at kA=kB=0.10)


,d0,reject,dk_rmse
0,0.5,0.10,0.003799
1,1.0,0.09,0.003681
2,2.0,0.08,0.003539
3,4.0,0.07,0.003393
4,8.0,0.06,0.003262


fitFDist → d0 = 6.96,  s0^2 = 2.37e-04


**Reading.** `fitFDist` returns a *sensible* d₀ (safe zone, moderated df well above the risky
band) but tends **more conservative** than the Type-I-optimal point on the sweep — it calibrates to
the variance noise, not to the Type-I/gain tradeoff. This is the gap to weigh when choosing between
**(a)** shipping a fixed d₀ (e.g. 2) and **(b)** the data-driven `fitFDist`.

## Next — real-data validation (no ground truth)

The sim has ground truth; real data does not. Plan (scaffold below): from the peptide-level
`riana_fit_fractions.txt` (now self-contained with `fs_var`/`fs_df` after the rollup re-run),
estimate **per-peptide k / Δk** and measure the **within-protein robust geometric CV** (the metric
from `reports/`) under `wls` vs `wls-var` — a lower within-protein CV = a more consistent estimator.
Also compare collapse vs pooled on the real data, per Scenario 2.

In [5]:
# SCAFFOLD (fill in): per-peptide k + within-protein robust geomCV on a real run.
# ff = pd.read_table("../runs/lve_atr_clean/riana_fit_fractions.txt", comment="#")
# ff["sigma"] = (ff.fs_upper - ff.fs_lower) / 3.29
# ... fit phi = -k t per (protein id, concat, condition); within-protein geomCV of k; wls vs wls-var.
print("TODO: real-data intra-protein geom-CV (design with more data, per the discussion)")

TODO: real-data intra-protein geom-CV (design with more data, per the discussion)
